# Getting Started with spharray

This notebook is a small, reproducible path through the package. It builds a spherical grid, creates a spherical-harmonic basis, checks a transform, evaluates a beamformer, and estimates a synthetic source direction. Run it from the repository root when working from a checkout.

## Setup

The next cell makes the local checkout importable even when the notebook server starts in a subdirectory. It then imports NumPy and the package.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / "spharray").is_dir():
        sys.path.insert(0, str(candidate))
        break

import spharray as sap
from spharray.sh import acn_index

print("package version:", sap.__version__)

## Build A Grid And SH Matrix

A spherical grid contains directions and quadrature weights. The SH matrix has one row per direction and `(N+1)^2` columns.

In [ ]:
order = 3
grid = sap.array.fibonacci_grid(2000)
spec = sap.SHBasisSpec(max_order=order, basis="complex", angle_convention="az_colat")
Y = sap.sh.matrix(spec, grid)

print("grid directions:", grid.size)
print("weights sum:", float(grid.weights.sum()))
print("SH matrix shape:", Y.shape)

## Transform A Known Harmonic

Here the input field is exactly one SH basis function. The forward transform should recover one dominant coefficient and near-zero values elsewhere. Fibonacci weights are approximate, so the error is small rather than exactly zero.

In [ ]:
target = np.zeros(spec.n_coeffs, dtype=np.complex128)
q = acn_index(2, 1)
target[q] = 1.0

samples = sap.sh.inverse_sht(target, Y)
recovered = sap.sh.direct_sht(samples, Y, grid)
error = float(np.max(np.abs(recovered - target)))

print("target channel:", q)
print("max coefficient error:", f"{error:.2e}")

## Evaluate A Fixed Beamformer

Fixed beamformers return one weight per SH degree. In this package they use unit front gain, so the response at `theta = 0` should be one.

In [ ]:
weights = sap.beamforming.beam_weights_hypercardioid(order)
theta = np.array([0.0, np.pi / 2.0, np.pi])
pattern = sap.beamforming.axisymmetric_pattern(theta, weights)

print("B(0 deg):", float(pattern[0]))
print("B(90 deg):", float(pattern[1]))
print("B(180 deg):", float(pattern[2]))

## Estimate A Synthetic Source Direction

This cell creates a rank-one covariance matrix for a known source direction and asks PWD and MUSIC to recover it on the same search grid.

In [ ]:
source_index = 137
search = sap.array.fibonacci_grid(600)
Y_search = sap.sh.matrix(spec, search)
steering = Y_search[source_index]
R = np.outer(steering, steering.conj()) + 0.01 * np.eye(spec.n_coeffs)

pwd = sap.doa.pwd_spectrum(R, search, spec, n_peaks=1)
music = sap.doa.music_spectrum(R, search, spec, n_sources=1, n_peaks=1)

print("true source index:", source_index)
print("PWD peak index:", int(pwd.peak_indices[0]))
print("MUSIC peak index:", int(music.peak_indices[0]))

## Next Steps

Read `docs/concepts.md` for the vocabulary behind these cells. Then run the scripts in `examples/tutorials/` to see the same ideas as command-line workflows that also perform numerical checks.